In [ ]:
import os
import avstack
import avapi
import cv2  
from IPython.display import Video


%load_ext autoreload
%autoreload 2

# Importing data
data_base = '../../data'
obj_data_dir_k = os.path.join(data_base, 'KITTI/object')
raw_data_dir_k = os.path.join(data_base, 'KITTI/raw') 
data_dir_n     = os.path.join(data_base, 'nuScenes')
data_dir_c     = os.path.join(data_base, 'CARLA/ego-lidar')

# Instantiating Scene Manager
KSM = avapi.kitti.KittiScenesManager(obj_data_dir_k, raw_data_dir_k, convert_raw=False)
NSM = avapi.nuscenes.nuScenesManager(data_dir_n)
CSM = avapi.carla.CarlaScenesManager(data_dir_c)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
def make_movie(DM, dataset_name, save_movie=False):
   
    model = '2d-img'
    if (dataset_name == 'nuscenes'):
       model = '3d-lidar'
       
    if model == '2d-img':
        M = avstack.modules.perception.object2dfv.MMDetObjectDetector2D(
            model='fasterrcnn', dataset=dataset_name, gpu=1)
    elif model == '3d-img':
        M = avstack.modules.perception.object3d.MMDetObjectDetector3D(
            model='pgd', dataset=dataset_name, gpu=0)
    elif model == '3d-lidar':
        M = avstack.modules.perception.object3d.MMDetObjectDetector3D(
            model='pointpillars', dataset=dataset_name, gpu=1)
    else:
        raise NotImplementedError(model)
    
    if DM.frames is None:
        print("NO FRAMES in Scene")
        return 
        
    range_frames = range(len(DM.frames))
    CAM = 'image-2'
    if (dataset_name == 'carla'):
        range_frames = range(4, len(DM.frames)-5)
        CAM = 'CAM_FRONT'
    if (dataset_name == 'nuscenes'):
        CAM = 'main_camera'        
        
    imgs = []
    for frame_idx in range_frames:
        frame = DM.get_frames(sensor="main_camera")[frame_idx]
        objects = DM.get_objects(frame, sensor='main_lidar')
        img = DM.get_image(frame, sensor=CAM)
        pc = DM.get_lidar(frame)
        if model == '2d-img':
            outputs = M(img, identifier='test', frame=frame)
        elif model == '3d-img':
            outputs = M(img, identifier='test', frame=frame)
        elif model == '3d-lidar':
            outputs = M(pc, identifier='test', frame=frame)
        else:
            raise NotImplementedError(model)
            
        imgs.append(avapi.visualize.snapshot.show_image_with_boxes(img, outputs.data, inline=False, return_images=True))

    # generate movie
    movie_name = dataset_name + '_scene_movie.mp4'
  
    height, width, layers = imgs[0].shape
    size = (width,height)
    video = cv2.VideoWriter(movie_name,cv2.VideoWriter_fourcc(*'DIVX'), 15, size)
    for img in imgs:
        video.write(img)  
    
    # Deallocating memories taken for window creation 
    cv2.destroyAllWindows()  
    print("Scene video:")
    
    video.release()  # releasing the video generated 

In [16]:
KDM = KSM.get_scene_dataset_by_index(scene_idx=0)
CDM = CSM.get_scene_dataset_by_index(scene_idx=0)
NDM = NSM.get_scene_dataset_by_index(scene_idx=0)
make_movie(NDM, 'nuscenes')

/home/judy/avdev-sandbox/submodules/lib-avstack-core/third_party/mmdetection3d/mmdet3d/models/dense_heads/anchor3d_head.py:92: UserWarning: dir_offset and dir_limit_offset will be depressed and be incorporated into box coder in the future
  warnings.warn(


Loads checkpoint by local backend from path: /home/judy/avdev-sandbox/submodules/lib-avstack-core/third_party/mmdetection3d/checkpoints/nuscenes/hv_pointpillars_fpn_sbn-all_fp16_2x8_2x_nus-3d_20201021_120719-269f9dd6.pth


Error: no "view" rule for type "image/png" passed its test case
       (for more information, add "--debug=1" on the command line)
/usr/bin/xdg-open: 882: www-browser: not found
/usr/bin/xdg-open: 882: links2: not found
/usr/bin/xdg-open: 882: elinks: not found
/usr/bin/xdg-open: 882: links: not found
/usr/bin/xdg-open: 882: lynx: not found
/usr/bin/xdg-open: 882: w3m: not found
xdg-open: no method available for opening '/tmp/tmpg8d0he8j.PNG'
Error: no "view" rule for type "image/png" passed its test case
       (for more information, add "--debug=1" on the command line)
/usr/bin/xdg-open: 882: www-browser: not found
/usr/bin/xdg-open: 882: links2: not found
/usr/bin/xdg-open: 882: elinks: not found
/usr/bin/xdg-open: 882: links: not found
/usr/bin/xdg-open: 882: lynx: not found
/usr/bin/xdg-open: 882: w3m: not found
xdg-open: no method available for opening '/tmp/tmpdjj_68lq.PNG'
Error: no "view" rule for type "image/png" passed its test case
       (for more information, add "--debug

Scene video:
